# 30 · Fine-tuned encoder baselines
Fine-tunes DeBERTa-v3-base and RoBERTa-base on each train split, selects the epoch by dev macro-F1, and predicts every test post label-blind (docs/12 §3.7). Scoring happens only in `99_final_eval` (post-hoc section).
- **Inputs:** `data/processed/{train,dev,test}`; CMSB annotated only; EDOS B/C trained on gold-sexist train posts; test is read label-blind (id and text only).
- **Outputs:** experiment folder `OUTPUT_ROOT/experiments/{YYYYMMDD-HHMMSS}_experiment_encoders/`, one run per (model, task, seed) in `runs/` with `config.json`, `predictions.jsonl` (test probabilities), `dev_predictions.jsonl`, `dev_metrics.json` (per-epoch dev macro-F1), `environment.json`, `complete.json`, `README.md`; `manifest.csv` and `heartbeat.json` at the top.
- **Fixed rules:** AdamW lr 2e-5, weight decay 0.01, 6% warm-up, batch 16, max length 128, 4 epochs (binary) / 6 (EDOS B/C), seeds 13, 42, 87. No tuning on test.
- **Sanity first:** `CONFIG["sample"] = N` trains on a few posts for a few steps into a separate `encoders-sample` experiment; check **SANITY OK**, then set `sample` to `None`.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Sexism-research")  # TODO: set to your Drive clone
else:
    PROJECT_ROOT = Path.cwd().parent  # local run from notebooks/
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"  # on Drive, so runs survive disconnects
HF_HOME = Path("/content/hf_cache") if IN_COLAB else PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)
sys.path.insert(0, str(PROJECT_ROOT))
if IN_COLAB:  # local copy of the processed data for speed; outputs still go to Drive
    shutil.copytree(DATA_ROOT / "processed", Path("/content/data/processed"), dirs_exist_ok=True)
    DATA_ROOT = Path("/content/data")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT  ", DATA_ROOT, "\nOUTPUT_ROOT", OUTPUT_ROOT, "\nHF_HOME    ", HF_HOME)

In [ ]:
# Same pins as 00 and requirements.txt; torch is Colab's build, logged not pinned
PINS = {"pandas": "3.0.6", "numpy": "2.5.4", "scikit-learn": "1.9.1", "pyarrow": "26.0.0", "pyyaml": "6.0.3",
        "transformers": "5.20.1", "pydantic": "2.13.5"}
if IN_COLAB:
    PIP_ARGS = " ".join(f"{p}=={v}" for p, v in PINS.items())
    %pip install -q {PIP_ARGS}
    print("If pip changed anything, restart the runtime, then run all again")
from importlib.metadata import version
from importlib.util import find_spec

off_pin = {p: version(p) for p in PINS if version(p) != PINS[p]}
assert not off_pin, f"installed {off_pin} differ from PINS: install the pins and restart the kernel"
# DeBERTa-v3's tokenizer is a SentencePiece model; Colab ships sentencepiece (TODO: pin it once approved)
print("sentencepiece available:", find_spec("sentencepiece") is not None)

In [ ]:
# The only cell to edit between batches
CONFIG = {
    "experiment": "encoders",
    "purpose": "Fine-tuned encoder reference on the same train/dev/test splits (docs/12 §3.7)",
    "protocol_ref": "docs/12-protocol-addendum.md §3.7",
    "sample": 5,  # sanity: N test posts, a few train/dev posts, 1 seed, 3 steps, into 'encoders-sample'; None = full
    "models": {"deberta-v3-base": "microsoft/deberta-v3-base", "roberta-base": "FacebookAI/roberta-base"},
    "tasks": [("cmsb", "binary"), ("edos", "task_a"), ("edos", "task_b"), ("edos", "task_c")],
    "seeds": [13, 42, 87],
    "epochs": {"binary": 4, "multiclass": 6},
    "hp": {"lr": 2e-5, "weight_decay": 0.01, "warmup": 0.06, "batch_size": 16, "max_length": 128},
    "heartbeat_s": 60,
}
SAMPLE = CONFIG["sample"] is not None

In [ ]:
# Token from Colab secrets (or the local env), never printed or pasted here
if IN_COLAB:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN set:", bool(HF_TOKEN))

In [ ]:
import json
import time
from datetime import datetime

import pandas as pd

from src.schemas import get_schema
from src.utils import (
    environment_info, experiment_date, load_split, make_run_name, open_experiment, prepare_run_dir,
    resolve_revision, runs_dir, sha256_file,
)

# Pin each encoder to a commit sha; a changed upstream sha changes CONFIG, so a resume is refused (new experiment)
REVISIONS = {name: resolve_revision(repo, "main", HF_TOKEN) for name, repo in CONFIG["models"].items()}
EXP_DIR = open_experiment(
    OUTPUT_ROOT, CONFIG["experiment"], {k: v for k, v in CONFIG.items() if k not in ("purpose", "protocol_ref")}
    | {"revisions": REVISIONS}, purpose=CONFIG["purpose"], protocol_ref=CONFIG["protocol_ref"],
    project_root=PROJECT_ROOT, sample=SAMPLE, ignore=("heartbeat_s",) + (("sample",) if SAMPLE else ()),
)
RUNS = runs_dir(EXP_DIR)
seeds = CONFIG["seeds"][:1] if SAMPLE else CONFIG["seeds"]
method = "finetune-test" + (f"-sample-n{CONFIG['sample']}" if SAMPLE else "")
JOBS = [(name, ds, task, seed, make_run_name(experiment_date(EXP_DIR), ds, task, name, method, seed))
        for name in CONFIG["models"] for ds, task in CONFIG["tasks"] for seed in seeds]
done = [j for j in JOBS if (RUNS / j[4] / "complete.json").exists()]
print(EXP_DIR, f"\n{len(JOBS)} jobs, {len(done)} done")

In [ ]:
from src.model.encoder import EncoderHP, train_encoder
from src.utils import write_heartbeat

t0 = time.time()
for k, (name, ds, task, seed, run_name) in enumerate(JOBS, 1):
    run_dir = RUNS / run_name
    if (run_dir / "complete.json").exists():
        continue
    write_heartbeat(EXP_DIR, {"state": "running", "n_jobs": len(JOBS), "current": run_name, "done": k - 1,
                              "elapsed_min": round((time.time() - t0) / 60, 1)})
    schema = get_schema(ds, task)
    binary = schema.task_type == "binary"
    classes = ["False", "True"] if binary else schema.labels()
    draw = {"train": CONFIG["sample"] * 8, "dev": CONFIG["sample"] * 4, "test": CONFIG["sample"]} if SAMPLE else {}
    frames, files = {}, {}
    for split in ("train", "dev", "test"):
        # Same filters as the LLM runs; test is label-blind (id, text_norm, split only)
        frames[split], files[split], _ = load_split(DATA_ROOT, ds, split, schema, sample=draw.get(split), seed=seed)
    hp = EncoderHP(**CONFIG["hp"], epochs=1 if SAMPLE else CONFIG["epochs"]["binary" if binary else "multiclass"],
                   max_steps=3 if SAMPLE else None)
    config = {
        "run_name": run_name, "kind": "encoder", "dataset": ds, "task": task, "split": "test", "model": name,
        "model_name": name, "model_id": CONFIG["models"][name], "revision": REVISIONS[name], "seed": seed,
        "method": "finetune", "hp": hp.__dict__, "classes": classes, "sample": CONFIG["sample"],
        "experiment": EXP_DIR.name, "n_examples": len(frames["test"]), "output_file": "predictions.jsonl",
        "data_files": {s: str(f.relative_to(DATA_ROOT)) for s, f in files.items()},
        "data_sha256": {s: sha256_file(f) for s, f in files.items()},
    }
    if run_dir.exists():  # an unfinished run is retrained from scratch (training is not resumable mid-way)
        for f in ("predictions.jsonl", "dev_predictions.jsonl", "dev_metrics.json"):
            (run_dir / f).unlink(missing_ok=True)
    prepare_run_dir(run_dir, config)
    (run_dir / "environment.json").write_text(json.dumps(environment_info(PROJECT_ROOT), indent=2))
    label_col = "sexist" if binary else schema.column
    out = train_encoder(CONFIG["models"][name], REVISIONS[name], frames["train"], frames["dev"], frames["test"],
                        classes, label_col, hp, seed, run_dir, HF_TOKEN)
    (run_dir / "complete.json").write_text(json.dumps({"n_predicted": len(frames["test"]),
                                                       "finished": datetime.now().isoformat(timespec="seconds"),
                                                       "best_epoch": out["best_epoch"]}, indent=2))
    (run_dir / "README.md").write_text(
        f"# {run_name}\n\n**Purpose.** Encoder baseline (docs/12 §3.7): {name} fine-tuned on {ds}/{task} train, "
        f"epoch chosen on dev (best dev macro-F1 {out['best_dev_macro_f1']:.3f} at epoch {out['best_epoch']}).\n"
        f"**Model.** `{CONFIG['models'][name]}` @ `{REVISIONS[name]}`; seed {seed}.\n"
        f"**Outcome.** Test probabilities in `predictions.jsonl` (label-blind); scored in `99_final_eval`.\n")
    print(f"[{k}/{len(JOBS)}] {run_name}: best dev macro-F1 {out['best_dev_macro_f1']:.3f} (epoch {out['best_epoch']}), "
          f"{out['train_minutes']} min")
write_heartbeat(EXP_DIR, {"state": "finished", "n_jobs": len(JOBS), "elapsed_min": round((time.time() - t0) / 60, 1)})

In [ ]:
from src.utils import sanity_table, write_manifest

MANIFEST = write_manifest(EXP_DIR)  # rebuilt from the run folders; 99 reads runs only through it
print(MANIFEST.status.value_counts().to_dict())
if SAMPLE:
    SANITY = sanity_table(EXP_DIR, [j[4] for j in JOBS])
    print("SANITY OK" if SANITY.ok.all() else f"SANITY FAILED: {int((~SANITY.ok).sum())} runs, see the table")
    display(SANITY)

## Summary
TODO after the full run: runs finished per model and task, best epochs, training time; test scores are only in `99_final_eval` (post-hoc section, `encoders_*` tables).